# Assignment 1: Centroid based clustering

### Make sure you read through this entire notebook before getting started with implementing the algorithm.


This notebook has the following structure:

- We first shortly explain the idea of the assignment.
- We follow thus up with a short walkthrough of the assignment, after which you can start implementing the necessary functions.


# Introduction to this template notebook

* This is a **personal** notebook.
* Make sure you work in a **copy** of `...-template.ipynb`,
**renamed** to `...-yourIDnr.ipynb`,
where `yourIDnr` is your TU/e identification number.

<div class="alert alert-danger" role="danger">
<h3>Integrity</h3>
<ul>
    <li>In this course you must act according to the rules of the TU/e code of scientific conduct.</li>
    <li>All the exercises and the graded assignments are to be executed individually and independently.</li>
    <li>You must not copy from the Internet, your friends, books... If you represent other people's work as your own, then that constitutes fraud and will be reported to the Examination Committee.</li>
    <li>Making your work available to others (complicity) also constitutes fraud.</li>
</ul>
</div>

You are expected to work with Python code in this notebook.

The locations where you should write your solutions can be recognized by
**marker lines**,
which look like this:

>`#//`
>    `BEGIN_TODO [Label]` `Description` `(n points)`
>
>`#//`
>    `END_TODO [Label]`

<div class="alert alert-warning" role="alert">Do NOT modify or delete these marker lines.  Keep them as they are.<br/>
<br/>
NEVER write code <i>outside</i> the marked blocks.
Such code cannot be evaluated.
</div>

Proceed in this notebook as follows:
* **Read** the text.
* **Fill in** your solutions between `BEGIN_TODO` and `END_TODO` marker lines.
* **Run** _all_ code cells (also the ones _without_ your code),
    _in linear order_ from the first code cell.

**Personalize your notebook**:
1. Copy the following three lines of code:

  ```python
  AUTHOR_NAME = 'Your Full Name'
  AUTHOR_ID_NR = '1234567'
  AUTHOR_DATE = 'YYYY-MM-DD'  # when notebook was first modified, e.g. '2020-02-26'
  ```

1. Paste them between the marker lines in the next code cell.
1. Fill in your _full name_, _identification number_, and the current _date_ as strings between quotes.
1. Run the code cell by putting the cursor there and typing **Control-Enter**.


In [15]:
#// BEGIN_TODO [Author] Name, Id.nr., Date, as strings (1 point)

AUTHOR_NAME = 'Your Full Name'
AUTHOR_ID_NR = '1234567'
AUTHOR_DATE = 'YYYY-MM-DD'

#// END_TODO [Author]

AUTHOR_NAME, AUTHOR_ID_NR, AUTHOR_DATE

('Your Full Name', '1234567', 'YYYY-MM-DD')

# Centroid based clustering

For this assignment, you are expected to implement the k_means clustering algorithm, as discussed in class. The functions you are expected to implement are
* ```initialize_centroids```. This function chooses the initial cluster points. Currently, this function already contains an implementation, however, you are strongly encouraged to experiment with creating different initialisation functions once you have implemented the ```cluster``` function.
* ```cluster```. This function should, once implemented by you, perform the actual clustering and find the proper placement of the centroids.

In addition to this file, you are provided two more python files:
* ```point.py```.  This file contains the datastructures ```Point```, ```ClusterPoint``` and ```CentroidPoint``` that you can use in your implementation of the two functions.
* ```dataset.py```. This file contains the tools that read the input from and write the output to file.

<b>Testing.</b> After (partially) implementing the two functions, you can run the <b>run</b> function at the bottom of this document. Just above this function, you are provided with the ```test_case_nr``` field. Changing this field to an integer value $x \in [0, 6]$ allows you to choose which testcase you would like to run. This then reads the file 0$x$.in from the <b>input</b> folder and provides the resulting clustering in 0$x$.out in the <b>output</b> folder.

<b>Visulisation.</b> To view the clustering you created, you can use the ```Visualizer.ipynb``` notebook that you are provided alongside this assignment. After executing the first cell, you can simply execute the cell that corresponds to the testcase you want to view and a visualisation is provided.

<b>Handing in.</b> To verify your implementation, you are expected to hand in this file on Canvas. Here, we use the automated checking tool Momotor to check your implementation of the algorithm. After it has run through all the testcases (should take at most a couple minutes), you can see in the Momotor tab of the course how you scored. 
Note: you should only hand in _this_ file. The visualizer and other python files should not be handed in.

We move on to the actual coding part of this assignment. At the start we import some useful tools.

In [16]:
import sys
import os
import time

from point import *
from dataset import *

""" Runtime parameters """
assignment_nr = 1       # The assignment number. Used by the visualizer to determine what has to be visualized

**Extra functions.** When implementing the ```cluster()``` and ```initialize-centroids()``` functions, you'll likely want to create a couple functions yourself. To make sure the automated grader picks up on these, make sure you place these functions in the below cell.

In [17]:
#// BEGIN_TODO [YOUR-OWN-FUNCTIONS] 

def compute_centroids(points: list[Point], k: int, d: int) -> list[CentroidPoint]:
    """
    Compute the optimal set of centroids.

    :param points: cluster points
    :param k: number of clusters
    :param d: dimension
    :return: optimal centroids.
    """
    n, cluster_sizes = len(points), [0] * k
    point_sum = [ClusterPoint(d) for i in range(k)]
    centroids = [CentroidPoint(d) for i in range(k)]

    for i in range(n):
        cluster_sizes[points[i].cluster_label] += 1
        point_sum[points[i].cluster_label].add(points[i])

    for i in range(k):
        point_sum[i].div(cluster_sizes[i])
        centroids[i] = point_sum[i]

    return centroids

def compute_labels(points: list[Point], centroids: list[Point]) -> list[Point]:
    """
    Compute assignment of labels.

    :param points: cluster points
    :param centroids: centroid points
    :return: list of points with labels.
    """
    n, k = len(points), len(centroids)
    
    for i in range(n):
        min_distance = float('inf')

        for j in range(k):
            current_distance = points[i].sq_distance_to(centroids[j])
            if current_distance < min_distance:
                min_distance = current_distance
                cluster_label = j

        points[i].cluster_label = cluster_label

    return points  # contains labels

def uniform_random():
    """
    Find x uniformly at random such that 0 < x < 1.
    """
    none = True

    while none:
        x = random.random()
    
        if 0 < x < 1:
            return x

#// END_TODO [YOUR-OWN-FUNCTIONS]

**Initialize centroids.** We continue with the ```intialize_centroids()``` function. This function currently already has a basic implementation: it takes the first $k$ cluster points from the input and sets these as initial centroids. When checking your algorithm implementations in Momotor, only this basic implementation will be used. However, for the report you are expected to write about this assignment, you are very much encouraged to overwrite this basic implementation with something different and report on your findings.

In [18]:
import random
import math

In [19]:
def initialize_centroids(input_obj):
    """
    Calculates the initial centroid placement

    :param input_obj:   the input object
    :return:            a list of k centroids in the plane
    """
    centroids = [CentroidPoint(p.dimension, list(p.coords)) for p in input_obj.cluster_points[:input_obj.k]]

#// BEGIN_TODO [IMPLEMENT-INITIALIZE-CENTROIDS]
    
    # k-means++
    points, n, d, k = input_obj.cluster_points, input_obj.n, input_obj.d, input_obj.k
    min_distances = [0] * n
    cumulative = [0] * n
    
    x = uniform_random()
    sampled_index = math.ceil(x * n)
    centroids[0] = points[sampled_index]

    for j in range(n):
        min_distances[j] = float('inf')

    for i in range(1, k):
        for j in range(n):
            x = points[j].sq_distance_to(centroids[i-1])

            if min_distances[j] > x:
                min_distances[j] = x

        cumulative[0] = min_distances[0] * min_distances[0]
        for j in range(1, n):
            cumulative[j] = cumulative[j-1] + (min_distances[j] * min_distances[j])
    
        x = uniform_random()
        x = x * cumulative[n-1]
    
        if x <= cumulative[0]:
            sampled_index = 0
        else:
            for j in range(1, n):
                if (x > cumulative[j-1]) and (x <= cumulative[j]):
                    sampled_index = j
    
        centroids[i] = points[sampled_index]

#// END_TODO [IMPLEMENT-INITIALIZE-CENTROIDS]
    return centroids

**Cluster.** Below we find the ```cluster()``` function. Currently, no implementations has been provided. It is your task to implement the k-means clustering algorithm here.

In [20]:
def cluster(input_obj):
    """
    Perform k-means clustering on the input set

    :param input_obj:   the input object
    :return:            a list of k centroids in the plane
    """    
#// BEGIN_TODO [IMPLEMENT-CLUSTER]    

    # Lloyd's algo
    centroids = initialize_centroids(input_obj)  # initialize centroids with first k points of input
    points, n, d, k = input_obj.cluster_points, input_obj.n, input_obj.d, input_obj.k
    different = True

    while different:
        labels = compute_labels(points, centroids)
        centroids_new = compute_centroids(points, k, d)

        if centroids_new == centroids:
            different = False
        
        centroids = centroids_new

#// END_TODO [IMPLEMENT-CLUSTER] 
    return centroids

Next, we define the function that will take the input from file, use your clustering algorithm on get the clustering and write the result to file again.

In [21]:
def run(path_in, path_out):
    """
    Reads the input set, clusters the points and writes to output

    :param path_in:     location of the input set
    :param path_out:    location to print the output
    """

    # read input from file
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return        

    # find the best centroids using k_means
    centroids = cluster(input_obj)

    # simple check if the correct number of centroids has been given
    assert len(centroids) == input_obj.k

    # print result to file
    try:
        input_obj.write_output(centroids, path_out, assignment_nr)
    except IOError:
        print("Could not write output to file: " + path_out, file=sys.stderr)

    print("Cluster counter:      ", input_obj.k, file=sys.stderr)
    print("Mean squared distance: {:.3f}".format(input_obj.avg_score(centroids)), file=sys.stderr)                

**Running testcases.** Lastly, you can check your implementation by running some tests on it. Below, you can choose which testcase ```test_case_nr``` $\in[0,6]$ you would like to run.

In [22]:
test_case_nr =   0     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       4
Mean squared distance: 137.727
Time taken:            0.014s


In [23]:
test_case_nr =   1     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       5
Mean squared distance: 29253.970
Time taken:            0.926s


In [24]:
test_case_nr =   2     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       4
Mean squared distance: 0.496
Time taken:            0.573s


In [25]:
test_case_nr =   3     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       7
Mean squared distance: 1.171
Time taken:            2.190s


In [26]:
test_case_nr =   4     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       6
Mean squared distance: 1.294
Time taken:            6.412s


In [27]:
test_case_nr =   5     # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       4
Mean squared distance: 0.197
Time taken:            0.227s


In [28]:
test_case_nr =   6    # which input file to read

if __name__ == "__main__":
    start = time.time()
    run("input/{:02d}.in".format(test_case_nr), "output/{:02d}.out".format(test_case_nr))
    end = time.time()
    print("Time taken:            {:.3f}s".format(end - start), file=sys.stderr)   

Cluster counter:       3
Mean squared distance: 34.756
Time taken:            1.612s


That is all. At this point you should have all the information you should need to get started. If you have any questions you are free to ask the tutor overseeing the class. If you are experiencing issues with handing in your submission to Momotor, you can contact the responsible teaching assistant via email. Their email address can be found on Canvas.

Best of luck and happy clustering!

&copy; 2019-2020 - **TU/e** - Eindhoven University of Technology